# Context engineering
Agents fail less often because the model is weak than because the wrong context reached it. Getting the right information, tools, and format into each call is the whole job.

## Concept
- Model context (transient) — what one call sees: prompt, messages, tools, model, output format.
- Tool context (persistent) — what tools read and write via state, store, runtime.
- Life-cycle context (persistent) — hooks between steps: summarize, guard, log.

## Shape the model call (transient)

In [3]:
from pydantic import SecretStr
import os

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import dynamic_prompt, ModelRequest


@dynamic_prompt
def state_aware_prompt(request: ModelRequest) -> str:
    base = "You are a helpful assistant."
    if len(request.messages) > 3:
        base += "\nThis is a long conversation — be concise."
    return base

model = init_chat_model(
    model=os.environ["FLEXAI_MODEL"],
    base_url=os.environ["FLEXAI_URL"],
    api_key=SecretStr(os.environ["FLEXAI_API_KEY"])
)

agent = create_agent(model=model, tools=[], middleware=[state_aware_prompt])

## Compress history between steps (persistent)

In [4]:
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model=model,
    tools=[],
    middleware=[SummarizationMiddleware(
        model=model,
        trigger={"tokens": 4000},
        keep=("messages", 20),
    )],
)

#### The dividing line: transient changes (via wrap_model_call, dynamic_prompt) affect one call only; persistent changes (life-cycle hooks, tool writes) are saved to state for every future turn.

## Read runtime context + state inside a tool

In [25]:
from typing import TypedDict

from langchain.tools import tool, ToolRuntime

class UserContext(TypedDict):
    user_id: str

@tool
def my_open_orders(runtime: ToolRuntime) -> str:
    """List the current user's open orders."""
    user_id = runtime.context.user_id                   # runtime context (static)
    authed = runtime.state.get("authenticated", False)  # state (short-term)
    if not authed:
        return "Please sign in first."
    return f"orders of user {user_id}: order1, order2, order3"

agent = create_agent(
    model=model,
    tools=[my_open_orders],
    context_schema=UserContext,
)

context = {"user_id": "user-123"}
response = agent.invoke(
    {"messages": [{"role": "user", "content": "fetch orders of user with id user-123"}]},
    context=context,
)
print(response["messages"][-1].content)

I’m only able to retrieve open orders for the currently logged‑in user. If you’d like to see the orders for the account you’re signed in with, just let me know and I’ll pull them up for you!


## Write to state with a Command
A tool can persist a fact for later steps — here, marking the session authenticated.

In [38]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def authenticate(password:str, runtime: ToolRuntime) -> Command:
    """authenticate the user with password"""
    is_correct = password == "correct"
    # Retrieve the tool_call_id from runtime so we can echo it back
    tool_call_id = runtime.tool_call_id
    
    status_text = "Authentication successful." if is_correct else "Incorrect password."
    return Command(update={
        # 1. Update the graph state
            "authenticated": is_correct,
            
            # 2. Provide the mandatory ToolMessage to satisfy message history
            "messages": [
                ToolMessage(content=status_text, tool_call_id=tool_call_id)
            ]
    })

agent = create_agent(
    model=model,
    tools=[authenticate, my_open_orders],
    context_schema=UserContext,
)

context = {"user_id": "user-123"}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "authenticate me with my password 'correct'.my user id is user-123. fetch orders after authentication"}]},
    context=context,
)
print(response["messages"][-1].content)

d:\nitmonk\agent stack\agent-stack-lab\.venv\Lib\site-packages\pydantic\functional_validators.py:851: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value={'user_id': 'user-123'}, input_type=dict])
  function=lambda v, h: h(v), schema=original_schema
d:\nitmonk\agent stack\agent-stack-lab\.venv\Lib\site-packages\pydantic\main.py:520: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value={'user_id': 'user-123'}, input_type=dict])
  return self.__pydantic_serializer__.to_python(
d:\nitmonk\agent stack\agent-stack-lab\.venv\Lib\site-packages\pydantic\functional_validators.py:851: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_valu

AttributeError: 'dict' object has no attribute 'user_id'

## Long-term memory via the store

The store survives across conversations — save a preference once, read it in any future thread.

In [39]:
@tool
def remember_preference(key: str, value: str, runtime: ToolRuntime) -> str:
    """Save a durable user preference."""
    user_id = runtime.context.user_id
    prefs = (runtime.store.get(("prefs",), user_id) or {}).value if \
            runtime.store.get(("prefs",), user_id) else {}
    prefs[key] = value
    runtime.store.put(("prefs",), user_id, prefs)
    return f"Saved {key} = {value}"

# Pick the model by conversation length
Route short chats to a cheap model, long ones to a big-context model — decided per call.

In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

big = init_chat_model("claude-sonnet-4-6")
small = init_chat_model("claude-haiku-4")

@wrap_model_call
def by_length(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    model = big if len(request.messages) > 20 else small
    return handler(request.override(model=model))

## Show tools only after sign-in
Trim the toolset by state — sensitive tools appear only once the user is authenticated.

In [ ]:
@wrap_model_call
def gate_tools(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    if not request.state.get("authenticated", False):
        public = [t for t in request.tools if t.name.startswith("public_")]
        request = request.override(tools=public)
    return handler(request)

## Adapt the response schema by stage
Return a simple shape early, a detailed one once the conversation matures.

In [ ]:
from pydantic import BaseModel, Field

class Simple(BaseModel):
    answer: str

class Detailed(BaseModel):
    answer: str
    reasoning: str
    confidence: float = Field(ge=0, le=1)

@wrap_model_call
def pick_format(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    schema = Detailed if len(request.messages) >= 3 else Simple
    return handler(request.override(response_format=schema))